In [38]:
# Imports
import boto3
from sagemaker import get_execution_role
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import joblib, os

In [2]:
role = get_execution_role()
bucket = 'json-avalos-ads508'
delog_data_key = 'Delivery_Logistics 2.csv'
dysulog_data_key = 'dynamic_supply_chain_logistics_dataset.csv'
trade_cus_data_key = 'trade_customs_dataset 2.csv'
logistics_data_key = 'Logistics_supply_chain.csv'
del_loc = 's3://{}/{}'.format(bucket, delog_data_key)
dyn_loc = 's3://{}/{}'.format(bucket, dysulog_data_key)
trade_loc = 's3://{}/{}'.format(bucket, trade_cus_data_key)
logistics_loc = 's3://{}/{}'.format(bucket, logistics_data_key)

In [3]:
!pip install openpyxl # dependency
delivery = pd.read_csv(del_loc)
dynamic = pd.read_csv(dyn_loc)
trade = pd.read_csv(trade_loc)
logistics = pd.read_csv(logistics_loc)

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)


In [4]:
def clean_trade(df: pd.DataFrame) -> pd.DataFrame:
    """trade_customs_dataset — parse dates, derive delay columns, basic QA."""
    df = df.copy()
    # Parse datetime columns
    for col in ["Shipment_Date", "Estimated_Arrival_Date", "Actual_Arrival_Date"]:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    # Drop rows where we can't compute the delay
    before = len(df)
    df = df.dropna(subset=["Estimated_Arrival_Date", "Actual_Arrival_Date"])
    print(f"  trade: dropped {before - len(df)} rows with missing arrival dates")
    # Derived delay features
    df["Transit_Delay_Days"] = (
        df["Actual_Arrival_Date"] - df["Estimated_Arrival_Date"]
    ).dt.days
    df["Is_Delayed"] = (df["Transit_Delay_Days"] > 0).astype(int)
    # Total delay = transit + customs
    if "Customs_Delay_Days" in df.columns:
        df["Total_Delay_Days"] = df["Transit_Delay_Days"] + df["Customs_Delay_Days"].fillna(0)
    # Temporal features
    df["Shipment_Month"] = df["Shipment_Date"].dt.month
    df["Shipment_DayOfWeek"] = df["Shipment_Date"].dt.dayofweek   # 0=Mon
    df["Shipment_IsWeekend"] = (df["Shipment_DayOfWeek"] >= 5).astype(int)
    # Normalise carrier / route strings
    for col in ["Carrier_Name", "Route_Code"]:
        if col in df.columns:
            df[col] = df[col].str.strip().str.upper()
    return df

In [5]:
def clean_delivery(df: pd.DataFrame) -> pd.DataFrame:
    """Delivery_Logistics — sanity checks, derived efficiency features."""
    df = df.copy()
 
    # Normalise all column names to lowercase for consistent access
    df.columns = df.columns.str.strip().str.lower()
 
    print(f"  delivery: columns found → {list(df.columns)}")
    print(f"  delivery: {len(df):,} rows on load")
 
    # ── Parse time columns — handle multiple possible formats ──────────────────
    def parse_time_to_hours(series: pd.Series, col: str) -> pd.Series:
        """
        Convert a time column to float hours regardless of format:
          "02:30"      -> 2.5
          "2:30:00"    -> 2.5
          "2.5"        -> 2.5
          "2.5 hrs"    -> 2.5
          "150"        -> 150.0  (assumed already in minutes if >24, else hours)
        """
        # Try plain numeric first
        attempt = pd.to_numeric(series, errors="coerce")
        if attempt.notna().mean() > 0.5:
            print(f"  delivery: '{col}' parsed as plain numeric")
            return attempt
 
        # Try HH:MM or HH:MM:SS string format
        def hhmm_to_hours(val):
            try:
                parts = str(val).strip().split(":")
                if len(parts) >= 2:
                    return int(parts[0]) + int(parts[1]) / 60
            except Exception:
                pass
            # Strip non-numeric suffixes like "hrs", "h", "min"
            try:
                import re
                cleaned = re.sub(r"[^\d.]", "", str(val))
                return float(cleaned) if cleaned else float("nan")
            except Exception:
                return float("nan")
 
        result = series.apply(hhmm_to_hours)
        n_converted = result.notna().sum()
        print(f"  delivery: '{col}' parsed from string format — "
              f"{n_converted:,}/{len(series):,} values converted, "
              f"sample raw values: {series.dropna().head(3).tolist()}")
        return result
 
    time_cols = ["expected_time_hours", "delivery_time_hours"]
    for col in time_cols:
        if col in df.columns:
            df[col] = parse_time_to_hours(df[col], col)
 
    # Coerce remaining numeric columns
    other_numeric = ["distance_km", "delivery_cost", "package_weight_kg", "delivery_rating"]
    for col in other_numeric:
        if col in df.columns:
            before_nulls = df[col].isna().sum()
            df[col] = pd.to_numeric(df[col], errors="coerce")
            new_nulls = df[col].isna().sum() - before_nulls
            if new_nulls > 0:
                print(f"  delivery: coerced {new_nulls} non-numeric values to NaN in '{col}'")
 
    # Guard against zero / null denominators — only filter if columns exist
    if "distance_km" in df.columns:
        before = len(df)
        df = df[df["distance_km"].notna() & (df["distance_km"] > 0)].copy()
        print(f"  delivery: dropped {before - len(df)} rows where distance_km <= 0 or null")
    else:
        print("  ⚠ delivery: 'distance_km' column not found — skipping filter")
 
    if "expected_time_hours" in df.columns:
        before = len(df)
        df = df[df["expected_time_hours"].notna() & (df["expected_time_hours"] > 0)].copy()
        print(f"  delivery: dropped {before - len(df)} rows where expected_time_hours <= 0 or null")
    else:
        print("  ⚠ delivery: 'expected_time_hours' column not found — skipping filter")
 
    # Derived features (only if source columns exist)
    if "delivery_time_hours" in df.columns and "expected_time_hours" in df.columns:
        df["time_vs_expected"] = df["delivery_time_hours"] / df["expected_time_hours"]
 
    if "delivery_cost" in df.columns and "distance_km" in df.columns:
        df["cost_per_km"] = df["delivery_cost"] / df["distance_km"]
 
    # Standardise the delay flag name (some versions use 'delayed', 'Delayed', 'is_delayed')
    delay_col = next((c for c in df.columns if c.lower() in ("delayed", "is_delayed")), None)
    if delay_col and delay_col != "delayed":
        df = df.rename(columns={delay_col: "delayed"})
 
    print(f"  delivery: {len(df):,} rows after cleaning, {df.shape[1]} cols")
    return df

In [6]:
def clean_dynamic(df: pd.DataFrame) -> pd.DataFrame:
    """dynamic_supply_chain_logistics_dataset — cast types, derive features."""
    df = df.copy()
    # Ensure congestion columns are numeric integers
    for col in ["port_congestion_level", "traffic_congestion_level"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    # Combined congestion index
    df["combined_congestion"] = (
        df.get("port_congestion_level", 0) + df.get("traffic_congestion_level", 0)
    )
    # Risk × clearance interaction
    if "route_risk_level" in df.columns and "customs_clearance_time" in df.columns:
        df["risk_clearance_product"] = (
            df["route_risk_level"] * df["customs_clearance_time"]
        )
    # Log-transform shipping costs (right-skewed)
    if "shipping_costs" in df.columns:
        df["log_shipping_costs"] = np.log1p(df["shipping_costs"])
    return df

In [7]:
def _iqr_bounds(series: pd.Series, k: float = 3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def clean_logistics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ── 1. Timestamp parsing + time features ─────────────────────────────────
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    df["year"] = df["timestamp"].dt.year
    df["month"] = df["timestamp"].dt.month
    df["day"] = df["timestamp"].dt.day
    df["day_of_week"] = df["timestamp"].dt.dayofweek   # 0 = Monday
    df["hour"] = df["timestamp"].dt.hour

    # ── 2. Binary target ──────────────────────────────────────────────────────
    df["is_delayed"] = (df["delivery_time_deviation"] > 0).astype(int)

    # ── 3. Operational risk indicators (thresholds from team notebook) ────────
    df["high_port_congestion"] = (df["port_congestion_level"] >= 7).astype(int)
    df["bad_weather"] = (df["weather_condition_severity"] >= 0.7).astype(int)
    df["low_supplier_reliability"]= (df["supplier_reliability_score"] <= 0.3).astype(int)

    # ── 4. Select final feature set ───────────────────────────────────────────
    keep = [
        "timestamp",
        "lead_time_days",
        "weather_condition_severity",
        "port_congestion_level",
        "supplier_reliability_score",
        "delivery_time_deviation",
        "year",
        "month",
        "day",
        "day_of_week",
        "hour",
        "is_delayed",
        "high_port_congestion",
        "bad_weather",
        "low_supplier_reliability",
    ]
    missing = [c for c in keep if c not in df.columns]
    if missing:
        print(f"  ⚠ logistics: expected columns not found: {missing}")
    df = df[[c for c in keep if c in df.columns]].copy()

    # ── 5. Validate ───────────────────────────────────────────────────────────
    print(f"  logistics: {len(df):,} rows, {df.shape[1]} cols, "
          f"is_delayed rate = {df['is_delayed'].mean():.2%}")
    n_missing = df.isnull().sum().sum()
    if n_missing:
        print(f"  logistics: {n_missing} nulls remaining (check timestamp parse)")

    return df

In [8]:
def merge_datasets(
    trade: pd.DataFrame,
    delivery: pd.DataFrame,
    dynamic: pd.DataFrame,
    logistics: pd.DataFrame,
) -> pd.DataFrame:
    """
    Returns a single merged DataFrame built in three layers:
 
    Layer 1 — trade LEFT JOIN dynamic on Route_Code (row-level, direct key).
    Layer 2 — delivery has no shared row-level key with trade, so aggregate
               its stats per delivery_mode and join as new columns.
    Layer 3 — logistics has no shared key either; join its aggregate stats
               per Route_Code if available, otherwise attach global stats.
 
    All joined columns are prefixed to avoid name collisions.
    """
 
    # ── Layer 1: trade + dynamic (row-level join) ─────────────────────────────
    if "Route_Code" in dynamic.columns:
        dynamic["Route_Code"] = dynamic["Route_Code"].astype(str).str.strip().str.upper()
        dyn_renamed = dynamic.rename(
            columns={c: f"dyn_{c}" for c in dynamic.columns if c != "Route_Code"}
        )
        merged = trade.merge(dyn_renamed, on="Route_Code", how="left")
        print(f"  Layer 1 — trade + dynamic on Route_Code "
              f"→ {merged.shape[0]:,} rows × {merged.shape[1]} cols")
    else:
        print("  ⚠ Layer 1 — no Route_Code in dynamic, skipping")
        merged = trade.copy()
 
    # ── Layer 2: delivery (aggregate per delivery_mode, join as new columns) ──
    # delivery has no shipment-level key shared with trade. Instead, compute
    # per-mode summary stats and join them onto trade rows via Transport_Mode,
    # which both datasets describe (air/road/sea etc.).
    delivery_join_key = None
    for trade_col, del_col in [("Transport_Mode", "delivery_mode")]:
        if trade_col in merged.columns and del_col in delivery.columns:
            delivery_join_key = (trade_col, del_col)
            break
 
    if delivery_join_key:
        trade_col, del_col = delivery_join_key
        # Normalise both sides to uppercase for matching
        delivery[del_col] = delivery[del_col].astype(str).str.strip().str.upper()
        merged[trade_col] = merged[trade_col].astype(str).str.strip().str.upper()
 
        agg_cols = {c: "mean" for c in ["time_vs_expected", "cost_per_km",
                                          "delivery_time_hours", "delivery_cost"]
                    if c in delivery.columns}
        if agg_cols:
            del_agg = (
                delivery.groupby(del_col)
                .agg(**{f"del_{c}_{fn}": (c, fn) for c, fn in agg_cols.items()})
                .reset_index()
                .rename(columns={del_col: trade_col})
            )
            merged = merged.merge(del_agg, on=trade_col, how="left")
            print(f"  Layer 2 — delivery stats joined via {trade_col} "
                  f"→ {merged.shape[1]} cols total")
    else:
        # No matching key at all — attach delivery global averages as scalar columns
        print("  ⚠ Layer 2 — no matching key for delivery, attaching global averages")
        for col in ["time_vs_expected", "cost_per_km"]:
            if col in delivery.columns:
                merged[f"del_global_{col}"] = delivery[col].mean()
 
    # ── Layer 3: logistics (aggregate per Route_Code or global) ───────────────
    shared_key = None
    for candidate in ["Route_Code", "route_code", "Shipment_ID", "shipment_id"]:
        if candidate in logistics.columns and candidate in merged.columns:
            shared_key = candidate
            break
 
    if shared_key:
        log_agg = logistics.groupby(shared_key).agg(
            log_avg_deviation=("delivery_time_deviation", "mean"),
            log_avg_lead_time=("lead_time_days", "mean"),
        ).reset_index()
        merged = merged.merge(log_agg, on=shared_key, how="left")
        print(f"  Layer 3 — logistics stats joined via {shared_key} "
              f"→ {merged.shape[1]} cols total")
    else:
        print("  ⚠ Layer 3 — no shared key for logistics, attaching global averages")
        for col in ["delivery_time_deviation", "lead_time_days"]:
            if col in logistics.columns:
                merged[f"log_global_{col}"] = logistics[col].mean()
 
    print(f"Final merged shape → {merged.shape[0]:,} rows × {merged.shape[1]} cols")
    return merged

In [9]:
trade = clean_trade(trade)
delivery = clean_delivery(delivery)
dynamic = clean_dynamic(dynamic)
logistics = clean_logistics(logistics)

  trade: dropped 0 rows with missing arrival dates
  delivery: columns found → ['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'distance_km', 'package_weight_kg', 'delivery_time_hours', 'expected_time_hours', 'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost']
  delivery: 25,000 rows on load
  delivery: 'expected_time_hours' parsed from string format — 25,000/25,000 values converted, sample raw values: ['1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000003', '1970-01-01 00:00:00.000000016']
  delivery: 'delivery_time_hours' parsed from string format — 25,000/25,000 values converted, sample raw values: ['1970-01-01 00:00:00.000000008', '1970-01-01 00:00:00.000000002', '1970-01-01 00:00:00.000000010']
  delivery: dropped 0 rows where distance_km <= 0 or null
  delivery: dropped 0 rows where expected_time_hours <= 0 or null
  delivery: 25,000 rows after cleaning, 17 cols
  logistics: 32,065 rows, 1

In [10]:
delivery.shape

(25000, 17)

In [11]:
merged = merge_datasets(trade, delivery, dynamic, logistics)

  ⚠ Layer 1 — no Route_Code in dynamic, skipping
  Layer 2 — delivery stats joined via Transport_Mode → 32 cols total
  ⚠ Layer 3 — no shared key for logistics, attaching global averages
Final merged shape → 10,000 rows × 34 cols


In [12]:
merged

,Shipment_ID,Origin_Country,Destination_Country,Shipment_Date,Estimated_Arrival_Date,Actual_Arrival_Date,Transport_Mode,Carrier_Name,Route_Code,Commodity_Type,...,Total_Delay_Days,Shipment_Month,Shipment_DayOfWeek,Shipment_IsWeekend,del_time_vs_expected_mean,del_cost_per_km_mean,del_delivery_time_hours_mean,del_delivery_cost_mean,log_global_delivery_time_deviation,log_global_lead_time_days
0,SHP000001,India,China,2022-01-31,2022-02-12,2022-02-15,AIR,CARRIER_9,R854,Electronics,...,6,1,0,0,NaN,NaN,NaN,NaN,5.177648,5.227502
1,SHP000002,Germany,Brazil,2024-08-31,2024-09-24,2024-09-25,SEA,CARRIER_36,R303,Machinery,...,2,8,5,1,NaN,NaN,NaN,NaN,5.177648,5.227502
2,SHP000003,China,Germany,2023-09-28,2023-10-13,2023-10-13,ROAD,CARRIER_10,R320,Automotive,...,0,9,3,0,NaN,NaN,NaN,NaN,5.177648,5.227502
3,SHP000004,Brazil,China,2024-02-25,2024-03-18,2024-03-21,SEA,CARRIER_25,R180,Food,...,6,2,6,1,NaN,NaN,NaN,NaN,5.177648,5.227502
4,SHP000005,India,China,2021-07-22,2021-08-20,2021-08-20,ROAD,CARRIER_6,R975,Pharmaceuticals,...,0,7,3,0,NaN,NaN,NaN,NaN,5.177648,5.227502
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,SHP009996,Japan,India,2023-03-24,2023-04-16,2023-04-16,RAIL,CARRIER_34,R839,Electronics,...,0,3,4,0,NaN,NaN,NaN,NaN,5.177648,5.227502
9996,SHP009997,South Africa,India,2024-02-22,2024-03-02,2024-03-05,AIR,CARRIER_48,R271,Pharmaceuticals,...,6,2,3,0,NaN,NaN,NaN,NaN,5.177648,5.227502
9997,SHP009998,USA,Brazil,2020-07-21,2020-08-12,2020-08-15,SEA,CARRIER_9,R635,Pharmaceuticals,...,6,7,1,0,NaN,NaN,NaN,NaN,5.177648,5.227502
9998,SHP009999,Australia,Brazil,2020-08-10,2020-09-08,2020-09-10,ROAD,CARRIER_50,R617,Pharmaceuticals,...,4,8,0,0,NaN,NaN,NaN,NaN,5.177648,5.227502


In [13]:
merged.columns

Index(['Shipment_ID', 'Origin_Country', 'Destination_Country', 'Shipment_Date',
       'Estimated_Arrival_Date', 'Actual_Arrival_Date', 'Transport_Mode',
       'Carrier_Name', 'Route_Code', 'Commodity_Type', 'Declared_Value_USD',
       'Weight_kg', 'HS_Code', 'Document_Status', 'Compliance_Score',
       'Prior_Offense_Count', 'Tariff_Category', 'Route_Risk_Index',
       'Inspection_Type', 'Delay_Reason', 'Customs_Delay_Days', 'Risk_Flag',
       'Transit_Delay_Days', 'Is_Delayed', 'Total_Delay_Days',
       'Shipment_Month', 'Shipment_DayOfWeek', 'Shipment_IsWeekend',
       'del_time_vs_expected_mean', 'del_cost_per_km_mean',
       'del_delivery_time_hours_mean', 'del_delivery_cost_mean',
       'log_global_delivery_time_deviation', 'log_global_lead_time_days'],
      dtype='object')

In [14]:
print("Shape:", merged.shape)
print()
print("Dtypes:")
print(merged.dtypes.value_counts())
print()
print("Columns:")
for c in merged.columns:
    print(f"  {c:<50s} {str(merged[c].dtype):<12s} nulls={merged[c].isna().sum()}")

Shape: (10000, 34)

Dtypes:
object            11
float64           11
int64              7
datetime64[ns]     3
int32              2
Name: count, dtype: int64

Columns:
  Shipment_ID                                        object       nulls=0
  Origin_Country                                     object       nulls=0
  Destination_Country                                object       nulls=0
  Shipment_Date                                      datetime64[ns] nulls=0
  Estimated_Arrival_Date                             datetime64[ns] nulls=0
  Actual_Arrival_Date                                datetime64[ns] nulls=0
  Transport_Mode                                     object       nulls=0
  Carrier_Name                                       object       nulls=0
  Route_Code                                         object       nulls=0
  Commodity_Type                                     object       nulls=0
  Declared_Value_USD                                 float64      nulls=0
  Weight_kg

In [29]:
CLF_TARGET = "Is_Delayed"          # binary classification
REG_TARGET = "Transit_Delay_Days"   # regression

# Columns that leak the target or carry no signal for ML
LEAKAGE_COLS = [
    "Actual_Arrival_Date",        # used to compute the delay — direct leakage
    "Transit_Delay_Days",         # leaks into CLF_TARGET
    "Total_Delay_Days",           # derived from Transit_Delay_Days
    "Shipment_Date",              # raw date — temporal info kept via extracted features
    "Estimated_Arrival_Date",
]

ID_COLS = ["Shipment_ID", "shipment_id", "Route_Code", "delivery_id"]
NAN_COLS = ["del_time_vs_expected_mean", "del_cost_per_km_mean", "del_delivery_time_hours_mean", "del_delivery_cost_mean"]

drop_cols = (
    [CLF_TARGET, REG_TARGET]
    + LEAKAGE_COLS
    + [c for c in ID_COLS if c in merged.columns] +
    NAN_COLS
)

feature_cols = [c for c in merged.columns if c not in drop_cols]
print(f"Features kept : {len(feature_cols)}")
print(f"Dropped : {[c for c in drop_cols if c in merged.columns]}")
print(f"Target (clf) : {CLF_TARGET}  — rate {merged[CLF_TARGET].mean():.2%}")
print(f"Target (reg) : {REG_TARGET}  — mean {merged[REG_TARGET].mean():.2f} days")

Features kept : 22
Dropped : ['Is_Delayed', 'Transit_Delay_Days', 'Actual_Arrival_Date', 'Transit_Delay_Days', 'Total_Delay_Days', 'Shipment_Date', 'Estimated_Arrival_Date', 'Shipment_ID', 'Route_Code', 'del_time_vs_expected_mean', 'del_cost_per_km_mean', 'del_delivery_time_hours_mean', 'del_delivery_cost_mean']
Target (clf) : Is_Delayed  — rate 63.07%
Target (reg) : Transit_Delay_Days  — mean 2.06 days


In [30]:
SPLIT_DATE = "2024-09-01"

df = merged.dropna(subset=[CLF_TARGET]).copy()
df = df.sort_values("Shipment_Date").reset_index(drop=True)

train_df = df[df["Shipment_Date"] <  SPLIT_DATE]
future_df = df[df["Shipment_Date"] >= SPLIT_DATE]
midpoint = len(future_df) // 2
val_df = future_df.iloc[:midpoint]
test_df = future_df.iloc[midpoint:]

print(f"Train : {len(train_df):>6,} rows  |  Is_Delayed={train_df[CLF_TARGET].mean():.2%}")
print(f"Val   : {len(val_df):>6,} rows  |  Is_Delayed={val_df[CLF_TARGET].mean():.2%}")
print(f"Test  : {len(test_df):>6,} rows  |  Is_Delayed={test_df[CLF_TARGET].mean():.2%}")

X_train_raw = train_df[feature_cols]
X_val_raw = val_df[feature_cols]
X_test_raw = test_df[feature_cols]

y_clf_train = train_df[CLF_TARGET].values
y_clf_val = val_df[CLF_TARGET].values
y_clf_test = test_df[CLF_TARGET].values

Train :  9,443 rows  |  Is_Delayed=63.08%
Val   :    278 rows  |  Is_Delayed=66.19%
Test  :    279 rows  |  Is_Delayed=59.50%


In [31]:
# ── Binary indicators (0/1) — pass through as-is ─────────────────────────
binary_cols = [
    c for c in feature_cols
    if (X_train_raw[c].dropna().isin([0, 1]).all() and X_train_raw[c].nunique() <= 2)
]

# ── High-cardinality categoricals — target encode ────────────────────────
high_card_cols = [
    c for c in X_train_raw.select_dtypes(include="object").columns
    if X_train_raw[c].nunique() > 10 and c not in binary_cols
]

# ── Low-cardinality categoricals — one-hot encode ────────────────────────
ohe_cols = [
    c for c in X_train_raw.select_dtypes(include="object").columns
    if X_train_raw[c].nunique() <= 10 and c not in binary_cols + high_card_cols
]

# ── Continuous cost/distance/weight — StandardScaler ─────────────────────
standard_cols = [
    c for c in feature_cols
    if any(kw in c.lower() for kw in ["cost", "distance", "weight", "value", "clearance"])
    and X_train_raw[c].dtype in ["float64", "int64", "float32", "int32"]
    and c not in binary_cols
]

# ── Bounded index/score columns — MinMaxScaler ────────────────────────────
minmax_cols = [
    c for c in feature_cols
    if any(kw in c.lower() for kw in [
        "congestion", "risk", "reliability", "severity",
        "time_vs_expected", "eta_variation", "rating"
    ])
    and X_train_raw[c].dtype in ["float64", "int64", "float32", "int32"]
    and c not in binary_cols + standard_cols
]

# ── Everything else numeric — impute only ────────────────────────────────
remaining_num = [
    c for c in X_train_raw.select_dtypes(include=np.number).columns
    if c not in binary_cols + standard_cols + minmax_cols
]

print("Column assignments")
print(f"  standard (StandardScaler) : {standard_cols}")
print(f"  minmax   (MinMaxScaler)   : {minmax_cols}")
print(f"  ohe      (OneHotEncoder)  : {ohe_cols}")
print(f"  high_card (TargetEncoder) : {high_card_cols}")
print(f"  binary   (passthrough)    : {binary_cols}")
print(f"  remaining numeric         : {remaining_num}")

Column assignments
  standard (StandardScaler) : ['Declared_Value_USD', 'Weight_kg']
  minmax   (MinMaxScaler)   : ['Route_Risk_Index', 'Risk_Flag']
  ohe      (OneHotEncoder)  : ['Origin_Country', 'Destination_Country', 'Transport_Mode', 'Commodity_Type', 'Document_Status', 'Tariff_Category', 'Inspection_Type', 'Delay_Reason']
  high_card (TargetEncoder) : ['Carrier_Name']
  binary   (passthrough)    : ['Shipment_IsWeekend']
  remaining numeric         : ['HS_Code', 'Compliance_Score', 'Prior_Offense_Count', 'Customs_Delay_Days', 'Shipment_Month', 'Shipment_DayOfWeek', 'log_global_delivery_time_deviation', 'log_global_lead_time_days']


In [32]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

try:
    import category_encoders as ce
    target_enc = ce.TargetEncoder(smoothing=1.0)
    hc_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", target_enc),
    ])
    print("Using TargetEncoder for high-cardinality columns")
except ImportError:
    # Frequency encoding fallback
    from sklearn.base import BaseEstimator, TransformerMixin
    class FrequencyEncoder(BaseEstimator, TransformerMixin):
        def fit(self, X, y=None):
            X = pd.DataFrame(X)
            self.freq_maps_ = {c: X[c].value_counts(normalize=True).to_dict() for c in X.columns}
            return self
        def transform(self, X):
            X = pd.DataFrame(X).copy()
            for c in X.columns:
                X[c] = X[c].map(self.freq_maps_.get(c, {})).fillna(0)
            return X.values
    hc_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", FrequencyEncoder()),
    ])
    print("Using FrequencyEncoder fallback (pip install category_encoders for TargetEncoder)")

transformers = []
if standard_cols : transformers.append(("std", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]),  standard_cols))
if minmax_cols   : transformers.append(("mm", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", MinMaxScaler())]),    minmax_cols))
if ohe_cols      : transformers.append(("ohe",    Pipeline([("imp", SimpleImputer(strategy="most_frequent")), ("enc", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), ohe_cols))
if high_card_cols: transformers.append(("hc",     hc_pipe,                                                                           high_card_cols))
if binary_cols   : transformers.append(("binary", SimpleImputer(strategy="most_frequent"),                                          binary_cols))
if remaining_num : transformers.append(("num",    SimpleImputer(strategy="median"),                                                  remaining_num))

preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=True,
)
print("Preprocessor built — not yet fitted")

Using FrequencyEncoder fallback (pip install category_encoders for TargetEncoder)
Preprocessor built — not yet fitted


In [27]:
# Fit on train — pass y for TargetEncoder (ignored by other transformers)
X_train = preprocessor.fit_transform(X_train_raw, y_clf_train)
X_val   = preprocessor.transform(X_val_raw)
X_test  = preprocessor.transform(X_test_raw)

print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"X_test  : {X_test.shape}")

# Recover feature names for inspection
try:
    feature_names_out = preprocessor.get_feature_names_out()
    print(f"Total output features: {len(feature_names_out)}")
except Exception:
    feature_names_out = None

X_train : (9443, 60)
X_val   : (278, 60)
X_test  : (279, 60)


/opt/conda/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['del_delivery_time_hours_mean']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['del_delivery_time_hours_mean']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/sklearn/impute/_base.py:653: UserWarning: Skipping features without any observed values: ['del_delivery_time_hours_mean']. At least one non-missing value is needed for imputation with strategy='most_frequent'.
  warnings.warn(


In [33]:
imbalance_rate = y_clf_train.mean()
print(f"Training class balance: {imbalance_rate:.2%} delayed")

try:
    from imblearn.over_sampling import SMOTE
    if imbalance_rate < 0.35:
        sm = SMOTE(sampling_strategy=0.5, random_state=42)
        X_train_bal, y_clf_train_bal = sm.fit_resample(X_train, y_clf_train)
        print(f"SMOTE applied: {len(y_clf_train):,} → {len(y_clf_train_bal):,} samples")
    else:
        X_train_bal, y_clf_train_bal = X_train, y_clf_train
        print("Classes sufficiently balanced — SMOTE not applied")
except ImportError:
    X_train_bal, y_clf_train_bal = X_train, y_clf_train
    print("imbalanced-learn not installed — skipping SMOTE (pip install imbalanced-learn)")

Training class balance: 63.08% delayed
imbalanced-learn not installed — skipping SMOTE (pip install imbalanced-learn)


In [37]:
os.makedirs("outputs", exist_ok=True)

# Save preprocessor locally only
joblib.dump(preprocessor, "outputs/preprocessor.pkl")
if feature_names_out is not None:
    pd.Series(feature_names_out).to_csv("outputs/feature_names.csv", index=False)
print("Saved: outputs/preprocessor.pkl")
print("Saved: outputs/feature_names.csv")

# Save train / val / test CSVs
for split_name, X, y in [
    ("train", X_train_bal, y_clf_train_bal),
    ("val",   X_val,       y_clf_val),
    ("test",  X_test,      y_clf_test),
]:
    cols = feature_names_out if feature_names_out is not None else range(X.shape[1])
    split_df = pd.DataFrame(X, columns=cols)
    split_df[CLF_TARGET] = y
    path = f"outputs/{split_name}.csv"
    split_df.to_csv(path, index=False)
    print(f"Saved: {path}  {split_df.shape}")

# Upload CSVs to S3 (pkl stays local only)
bucket = "json-avalos-ads508"
prefix = "project-processed-data"
s3_files = ["train.csv", "val.csv", "test.csv"]

s3 = boto3.client("s3")
for fname in s3_files:
    local_path = f"outputs/{fname}"
    s3_key = f"{prefix}/{fname}"
    try:
        s3.upload_file(local_path, bucket, s3_key)
        print(f"Uploaded: s3://{bucket}/{s3_key}")
    except Exception as e:
        print(f"FAILED {fname}: {e}")

Saved: outputs/preprocessor.pkl
Saved: outputs/feature_names.csv
Saved: outputs/train.csv  (9443, 61)
Saved: outputs/val.csv  (278, 61)
Saved: outputs/test.csv  (279, 61)
FAILED feature_names.csv: [Errno 2] No such file or directory: 'outputs/feature_names.csv'
Uploaded: s3://json-avalos-ads508/project-processed-data/train.csv
Uploaded: s3://json-avalos-ads508/project-processed-data/val.csv
Uploaded: s3://json-avalos-ads508/project-processed-data/test.csv
